In [ ]:
import importlib.util, subprocess, sys
_missing = [p for p in ('huggingface_hub', 'tqdm', 'pandas', 'matplotlib') if importlib.util.find_spec(p) is None]
if _missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing], check=True)
    print('installed', _missing, '- now Restart the kernel once, then Run All')
else:
    print('all packages present')

In [ ]:
import os, re, json, math, random, hashlib, zipfile, gc, time, pickle, warnings, sys, copy, shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

SMOKE_TEST = False
SEED = 42
RESUME_COMPLETED_RUNS = True
WRITE_SUBMISSIONS = True
RUN_LENGTH_STUDY = True
RUN_SOURCE_STUDY = True

DATASET_ID = "Exploration-Lab/CS-781-Capstone"
DATA_SOURCE = "hf"
LOCAL_DATA_DIR = None
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", "."]
PHASE1_CKPT_OVERRIDE = None
TOKENIZER_PATH_OVERRIDE = None

SOURCE_TOKENS = 768
TARGET_TOKENS = 384
SOURCE_STRATEGY = "salient"

VAL_DOCS = 150
VAL_GEN_DOCS = 60
XL_MEMORY = 0
XL_MEMORY_OLD = 0

GEN_MIN_TOKENS = 120
NO_REPEAT_NGRAM = 3
REPETITION_PENALTY = 1.2
GEN_BATCH = 16

EPOCHS = 3
MAX_TRAIN_MINUTES = 35
BATCH_DOCS = 4
GRAD_ACCUM = 4
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
GRAD_CLIP = 1.0
BACKBONE_DROPOUT = 0.1
USE_GRAD_CHECKPOINTING = False

EXPERIMENT_LIBRARY = {
    "lorma_plus": dict(method="lorma_plus", rank=16, alpha=16, lr=5e-4, patience=2,
                       description="LoRMA+ (compulsory): multiplicative low-rank update W = (I + a/r * B A) W0."),
    "lora": dict(method="lora", rank=16, alpha=32, lr=5e-4, patience=2,
                 description="LoRA: additive low-rank update W0 + (a/r) B A."),
    "qlora": dict(method="qlora", rank=16, alpha=32, lr=5e-4, patience=2,
                  description="QLoRA: frozen Phase-1 weights stored in 4-bit NF4 (block 64, absmax) + LoRA on top."),
    "adalora": dict(method="adalora", init_rank=12, target_rank=4, alpha=16, orth_reg=0.1, beta1=0.85, beta2=0.85,
                    t_init=0.10, t_final=0.20, delta_t=10, lr=8e-4, patience=2,
                    description="AdaLoRA: SVD-form update P diag(E) Q with an importance-driven rank budget (12 -> 4 avg)."),
    "houlsby": dict(method="houlsby", bottleneck=64, lr=8e-4, patience=2,
                    description="Houlsby adapters: bottleneck MLP after every attention output and every FFN (+ norms)."),
    "pfeiffer": dict(method="pfeiffer", bottleneck=128, lr=8e-4, patience=2,
                     description="Pfeiffer adapters: one bottleneck MLP after the FFN of every layer (+ norms)."),
    "prefix": dict(method="prefix", prefix_len=32, prefix_hidden=256, lr=8e-4, patience=2,
                   description="Prefix tuning: trainable per-layer prefix states (MLP re-parameterised) in every layer's memory."),
    "full": dict(method="full", lr=3e-5, patience=2, description="Full fine-tuning (reference)."),
    "zero_shot": dict(method="zero_shot", lr=0.0, patience=1, epochs=0,
                      description="No training: the Phase-1 model continues the prompt (how useful is Phase-1 alone?)."),
    "distill": dict(method="distill", kd_temperature=2.0, kd_alpha=0.5, lr=5e-5, patience=2,
                    description="Word-level knowledge distillation: fine-tuned teacher -> smaller student."),
}
EXPERIMENT_DEFAULTS = dict(source_strategy=None, epochs=None, dropout=None, use_amp=True)

def make_experiment(base, **over):
    cfg = copy.deepcopy(EXPERIMENT_LIBRARY[base])
    cfg.update(over)
    return cfg

ARCH_NAME = "transformer_xl"
OUTPUT_NAME = "phase6_transformer_xl_summ"
MIN_PARAMS, MAX_PARAMS = 50e6, 300e6
PHASE1_ARCH_NAME = "transformer_xl_relative_recurrence_tuned"
LOWRANK_TARGETS = ("attn.q_net", "attn.k_net", "attn.v_net", "attn.o_net", "ff.net.0", "ff.net.2")
FFN_OUT_TARGETS = ("ff.net.2",)
ADAPTER_TARGETS = ("attn.o_net", "ff.net.2")
EXPERIMENTS = {
    "E1_lorma_plus": make_experiment("lorma_plus"),
    "E4_distill": make_experiment("distill", student_layers=6,
                                  description="Word-level KD: best fine-tuned 12-layer teacher (~99.8M) -> 6-layer student (~53.8M)."),
}
RUN_LIST = ["E1_lorma_plus", "E4_distill"]

OUTPUT_DIR = Path("/kaggle/working") / OUTPUT_NAME if Path("/kaggle/working").exists() else Path(OUTPUT_NAME)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "runs").mkdir(exist_ok=True)

if SMOKE_TEST:
    SOURCE_TOKENS, TARGET_TOKENS, VAL_DOCS, VAL_GEN_DOCS, GEN_MIN_TOKENS = 64, 24, 6, 4, 4
    BATCH_DOCS, GRAD_ACCUM, EPOCHS, GEN_BATCH = 2, 1, 2, 4
    XL_MEMORY = min(XL_MEMORY, 32)
    XL_MEMORY_OLD = min(XL_MEMORY_OLD, 32)
    MIN_PARAMS = 0
    for _cfg in EXPERIMENTS.values():
        if "delta_t" in _cfg:
            _cfg["delta_t"] = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          "| memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
elif not SMOKE_TEST:
    raise RuntimeError("Enable a GPU accelerator (Kaggle: Settings > Accelerator > GPU T4/P100).")
print("Experiments to run:", RUN_LIST)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

def save_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)

def make_grad_scaler(enabled):
    if not enabled:
        return None
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=True)

In [ ]:
class ByteBPE:
    EOW = 260

    def __init__(self):
        self.specials = ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos, self.stoi, self.payload, self.merges = [], {}, {}, []
        self.vocab_size = 0
        self._ranks = {}
        self._cache = {}

    @classmethod
    def from_json(cls, payload):
        t = cls()
        t.itos = t.specials + [f"<0x{i:02X}>" for i in range(256)] + ["</w>"]
        t.stoi = {s: i for i, s in enumerate(t.itos)}
        t.payload = {i: bytes([i - 4]) for i in range(4, 260)}
        t.payload[260] = b""
        t.merges = [tuple(map(int, p)) for p in payload["merges"]]
        for pair in t.merges:
            new_id = len(t.itos)
            symbol = f"<{pair[0]}+{pair[1]}>"
            t.itos.append(symbol)
            t.stoi[symbol] = new_id
            t.payload[new_id] = t.payload.get(pair[0], b"") + t.payload.get(pair[1], b"")
        t.vocab_size = len(t.itos)
        if t.itos != payload.get("itos"):
            raise ValueError("Saved ByteBPE vocabulary and merge list are inconsistent.")
        t._ranks = {pair: r for r, pair in enumerate(t.merges)}
        return t

    def encode_word_reference(self, word):
        seq = [b + 4 for b in str(word).encode("utf-8")] + [self.EOW]
        for rank, pair in enumerate(self.merges):
            out, j = [], 0
            while j < len(seq):
                if j + 1 < len(seq) and (seq[j], seq[j + 1]) == pair:
                    out.append(261 + rank); j += 2
                else:
                    out.append(seq[j]); j += 1
            seq = out
        return seq

    def encode_piece(self, piece, add_eow=True):
        key = (piece, add_eow)
        hit = self._cache.get(key)
        if hit is not None:
            return hit
        seq = [b + 4 for b in piece.encode("utf-8")]
        if add_eow:
            seq.append(self.EOW)
        ranks = self._ranks
        while len(seq) > 1:
            best_rank, best_pair = None, None
            for i in range(len(seq) - 1):
                r = ranks.get((seq[i], seq[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_pair = r, (seq[i], seq[i + 1])
            if best_pair is None:
                break
            out, j, new_id = [], 0, 261 + best_rank
            while j < len(seq):
                if j + 1 < len(seq) and (seq[j], seq[j + 1]) == best_pair:
                    out.append(new_id); j += 2
                else:
                    out.append(seq[j]); j += 1
            seq = out
        if len(self._cache) < 400_000:
            self._cache[key] = seq
        return seq

def _sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def _search(patterns, roots):
    found, seen = [], set()
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pat in patterns:
            for p in sorted(root.rglob(pat)):
                rp = str(p.resolve())
                if p.is_file() and rp not in seen:
                    seen.add(rp); found.append(p)
    return found

if PHASE1_CKPT_OVERRIDE:
    ckpt_candidates = [Path(PHASE1_CKPT_OVERRIDE)]
else:
    ckpt_candidates = _search(["best_checkpoint.pt", "best*checkpoint*.pt"], SEARCH_ROOTS) + \
                      _search(["latest*checkpoint*.pt", "*.pt"], SEARCH_ROOTS)
_payload, PHASE1_CKPT_PATH = None, None
for cand in ckpt_candidates:
    try:
        pl = torch.load(cand, map_location="cpu", weights_only=False)
    except Exception as exc:
        print(f"  skip {cand}: {exc}"); continue
    arch = str(pl.get("architecture", "")) if isinstance(pl, dict) else ""
    if isinstance(pl, dict) and "model_state_dict" in pl and arch == PHASE1_ARCH_NAME:
        _payload, PHASE1_CKPT_PATH = pl, cand
        break
    print(f"  skip {cand}: architecture={arch!r}")
    del pl
if _payload is None:
    raise FileNotFoundError("No standard Transformer-XL Phase-1 checkpoint (architecture transformer_xl_relative_recurrence_tuned) "
                            "found. Attach the transformer_xl_tuned_100m output with best_checkpoint.pt, or set PHASE1_CKPT_OVERRIDE.")
print("Phase-1 checkpoint:", PHASE1_CKPT_PATH)
P1_STATE = _payload["model_state_dict"]
P1_CFG = dict(_payload["model_config"])
P1_VOCAB = int(_payload.get("vocab_size", P1_CFG["vocab_size"]))
P1_TOKENIZER_SHA = _payload.get("tokenizer_sha256")
P1_ARCH = _payload.get("architecture")
P1_CFG.setdefault("rel_base", 10_000.0)
print("Phase-1 architecture:", P1_ARCH)
print("Phase-1 config:", {k: P1_CFG.get(k) for k in ("d_model", "heads", "layers", "ff_hidden", "training_segment_len",
                                                     "training_mem_len", "vocab_size")})
del _payload
gc.collect()

if TOKENIZER_PATH_OVERRIDE:
    tok_candidates = [Path(TOKENIZER_PATH_OVERRIDE)]
else:
    tok_candidates = _search(["legal_bpe_tokenizer*.json", "*bpe*.json", "*tokenizer*.json"], SEARCH_ROOTS)

TOKENIZER, TOKENIZER_PATH, TOKENIZER_SHA256 = None, None, None
tok_table = []
for cand in tok_candidates:
    try:
        raw = json.loads(cand.read_text(encoding="utf-8"))
        if not (isinstance(raw, dict) and "merges" in raw and "itos" in raw):
            continue
        sha = _sha256_file(cand)
        tok_table.append((cand, int(raw.get("vocab_size", len(raw["itos"]))), sha, raw))
    except Exception:
        continue

print("Tokenizer files found:")
for cand, vs, sha, _ in tok_table:
    print(f"  {cand}  vocab={vs}  sha256={sha[:12]}")

chosen = [t for t in tok_table if P1_TOKENIZER_SHA and t[2] == P1_TOKENIZER_SHA]
how = "sha256 match with the Phase-1 checkpoint"
if not chosen:
    chosen = [t for t in tok_table if t[1] == P1_VOCAB]
    how = f"vocab size match ({P1_VOCAB}); sha256 differs/unknown - verify this is your Phase-1 tokenizer file"
if not chosen:
    raise FileNotFoundError(
        f"No tokenizer with vocab size {P1_VOCAB} found. Attach your legal_bpe_tokenizer JSON for that size "
        "(or set TOKENIZER_PATH_OVERRIDE). This notebook never trains a tokenizer.")
TOKENIZER_PATH, _, TOKENIZER_SHA256, _raw = chosen[0]
TOKENIZER = ByteBPE.from_json(_raw)
del tok_table, _raw
print(f"Using tokenizer: {TOKENIZER_PATH} ({how})")

PAD_ID, UNK_ID, BOS_ID, EOS_ID = (TOKENIZER.stoi[s] for s in ("<pad>", "<unk>", "<s>", "</s>"))
EOW_ID = ByteBPE.EOW
VOCAB_SIZE = TOKENIZER.vocab_size
assert VOCAB_SIZE == P1_VOCAB, f"Tokenizer vocab {VOCAB_SIZE} != checkpoint vocab {P1_VOCAB}"
print("Vocab size:", VOCAB_SIZE, "| ids:", dict(pad=PAD_ID, unk=UNK_ID, bos=BOS_ID, eos=EOS_ID, eow=EOW_ID))

_rng = random.Random(0)
_words = ["The", "Supreme", "Court", "of", "India,", "Section", "302", "IPC.", "(1995)", "Crl.A.No.123/2005",
          "appellant-State", "Hon'ble", "₹5,000", "naïve"]
for _w in _words:
    assert TOKENIZER.encode_piece(_w, True) == TOKENIZER.encode_word_reference(_w), _w
print("Fast BPE encoder == Phase-1 reference encoder on self-check words.")

_ENDS_WORD = np.zeros(TOKENIZER.vocab_size, dtype=bool)
_ENDS_WORD[ByteBPE.EOW] = True
for _r, (_a, _b) in enumerate(TOKENIZER.merges):
    _ENDS_WORD[261 + _r] = _ENDS_WORD[_b]
_SPECIAL = {PAD_ID, UNK_ID, BOS_ID, EOS_ID}

def decode_ids(ids):
    out = bytearray()
    for i in ids:
        i = int(i)
        if i in _SPECIAL:
            continue
        out += TOKENIZER.payload.get(i, b"")
        if _ENDS_WORD[i]:
            out += b" "
    return out.decode("utf-8", errors="ignore").strip()

def encode_text(text, limit=None):
    ids = []
    for w in text.split():
        ids.extend(TOKENIZER.encode_piece(w, True))
        if limit and len(ids) >= limit:
            return ids[:limit]
    return ids

_probe_txt = "The appeal is allowed and the order of the High Court dated 12.03.2001 is set aside."
assert decode_ids(encode_text(_probe_txt)) == _probe_txt, decode_ids(encode_text(_probe_txt))
print("decode(encode(text)) == text on the self-check sentence.")

PROMPT_IDS = [EOS_ID] + encode_text("Summary:")
print(f"Transformer-XL: source memory {SOURCE_TOKENS} tokens -> summary {TARGET_TOKENS} tokens "
      f"(Phase-1 memory {P1_CFG.get('training_mem_len')}, segment {P1_CFG.get('training_segment_len')})")
SOURCE_PAD_SIDE = "left"

In [ ]:
SPLIT_FILES = {"train": "train.json", "test": "test.json"}
PHASE_RE = r"phase[_ -]?6|summ"

def _pick(hits):
    pref = [h for h in hits if re.search(PHASE_RE, str(h), re.I)]
    return (pref or hits)[0]

def locate_phase6_files():
    paths = {}
    if DATA_SOURCE == "local":
        roots = [LOCAL_DATA_DIR] if LOCAL_DATA_DIR else SEARCH_ROOTS
        for split, name in SPLIT_FILES.items():
            hits = _search([name], roots)
            if not hits:
                raise FileNotFoundError(f"{name} not found under {roots}")
            paths[split] = _pick(hits)
        return paths
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        from getpass import getpass
        token = getpass("Hugging Face read token: ")
    from huggingface_hub import HfApi, hf_hub_download
    repo_files = HfApi().list_repo_files(DATASET_ID, repo_type="dataset", token=token)
    for split, name in SPLIT_FILES.items():
        hits = [f for f in repo_files if f.endswith("/" + name) or f == name]
        ph = [f for f in hits if re.search(PHASE_RE, f, re.I)]
        if not ph:
            raise FileNotFoundError(f"Phase-6 {name} not found in {DATASET_ID}. Candidates: {hits[:10]} | "
                                    f"some files: {repo_files[:30]}")
        paths[split] = Path(hf_hub_download(DATASET_ID, ph[0], repo_type="dataset", token=token))
    return paths

P6_PATHS = locate_phase6_files()
for k, v in P6_PATHS.items():
    print(f"{k:5s} -> {v}")

def _records(obj):
    if isinstance(obj, dict):
        for key in ("data", "documents", "docs", "examples"):
            if key in obj and isinstance(obj[key], list):
                return obj[key]
        return [dict(v, id=v.get("id", k)) if isinstance(v, dict) else {"id": k, "document": v} for k, v in obj.items()]
    return obj

def _text(v):
    if isinstance(v, list):
        return " ".join("" if t is None else str(t) for t in v)
    return "" if v is None else str(v)

def load_split(path, with_summary):
    raw = path.read_text(encoding="utf-8")
    try:
        obj = json.loads(raw)
    except json.JSONDecodeError:
        obj = [json.loads(line) for line in raw.splitlines() if line.strip()]
    docs = []
    for i, d in enumerate(_records(obj)):
        doc = {"id": d.get("id", d.get("doc_id", f"doc_{i}")),
               "document": _text(d.get("document", d.get("text", d.get("judgment", ""))))}
        if with_summary:
            s = d.get("summary", d.get("reference", d.get("abstract")))
            if s is None:
                raise KeyError(f"{doc['id']}: no summary field; keys = {list(d)}")
            doc["summary"] = _text(s)
        docs.append(doc)
    return docs

_all_train = load_split(P6_PATHS["train"], with_summary=True)
_rng = random.Random(SEED)
_idx = list(range(len(_all_train))); _rng.shuffle(_idx)
_nval = min(VAL_DOCS, max(1, len(_all_train) // 10))
DATA = {"train": [_all_train[i] for i in sorted(_idx[_nval:])], "val": [_all_train[i] for i in sorted(_idx[:_nval])],
        "test": load_split(P6_PATHS["test"], with_summary=False)}
test_ids = [d["id"] for d in DATA["test"]]
assert len(set(map(str, test_ids))) == len(test_ids), "duplicate test ids"
for s, docs in DATA.items():
    w = np.array([len(d["document"].split()) for d in docs])
    extra = ""
    if s != "test":
        sw = np.array([len(d["summary"].split()) for d in docs])
        extra = f" | summary words: median {int(np.median(sw)):,} p90 {int(np.percentile(sw, 90)):,} | compression x{np.median(w / np.maximum(sw, 1)):.1f}"
    print(f"[{s:5s}] {len(docs):5,d} documents | document words: median {int(np.median(w)):,} p90 {int(np.percentile(w, 90)):,}{extra}")
print("(no public dev split in Phase 6 -> 'val' is held out from train.json)")

In [ ]:
_SENT_RE = re.compile(r"(?<=[.?!;:])\s+(?=[A-Z0-9(\"'“])")
_WORD_RE = re.compile(r"[a-z0-9]+")

def split_sentences(text):
    s = [x.strip() for x in _SENT_RE.split(text) if x.strip()]
    out = []
    for x in s:
        w = x.split()
        out.extend(" ".join(w[i:i + 60]) for i in range(0, len(w), 60))
    return out

_df = Counter()
for d in DATA["train"]:
    _df.update(set(_WORD_RE.findall(d["document"].lower())))
_NDOC = max(1, len(DATA["train"]))
IDF = {w: math.log((1 + _NDOC) / (1 + c)) + 1.0 for w, c in _df.items()}
_IDF_DEFAULT = math.log(1 + _NDOC) + 1.0

def _tfidf(words):
    c = Counter(words)
    v = {w: n * IDF.get(w, _IDF_DEFAULT) for w, n in c.items()}
    norm = math.sqrt(sum(x * x for x in v.values())) or 1.0
    return {w: x / norm for w, x in v.items()}

def select_source(text, budget, strategy):
    if strategy == "head":
        return encode_text(text, budget)
    if strategy == "head+tail":
        words = text.split()
        head = encode_text(" ".join(words[:budget]), budget // 3)
        tail, rest = [], budget - len(head)
        for w in reversed(words[len(words) // 3:]):
            p = TOKENIZER.encode_piece(w, True)
            if len(tail) + len(p) > rest:
                break
            tail = p + tail
        return head + tail
    sents = split_sentences(text)
    if not sents:
        return [UNK_ID]
    toks_words = [_WORD_RE.findall(s.lower()) for s in sents]
    centroid = _tfidf([w for ws in toks_words for w in ws])
    n = len(sents)
    scores = []
    for i, ws in enumerate(toks_words):
        v = _tfidf(ws) if ws else {}
        cos = sum(x * centroid.get(w, 0.0) for w, x in v.items())
        rel = i / max(1, n - 1)
        bonus = 0.10 if rel < 0.10 else (0.15 if rel > 0.85 else 0.0)
        scores.append(cos + bonus - (0.05 if len(ws) < 6 else 0.0))
    chosen, used = [], 0
    for i in np.argsort(-np.array(scores)):
        ids = encode_text(sents[i])
        if used + len(ids) > budget:
            if not chosen:
                chosen.append((i, ids[:budget])); used = budget
            continue
        chosen.append((i, ids)); used += len(ids)
        if used >= budget - 8:
            break
    return [t for _, ids in sorted(chosen) for t in ids] or [UNK_ID]

def build_inputs(docs, strategy, with_target):
    out = []
    for d in tqdm(docs, desc=f"inputs ({strategy})", leave=False):
        it = {"id": d["id"], "src": np.asarray(select_source(d["document"], SOURCE_TOKENS, strategy), dtype=np.int32)}
        if with_target:
            it["tgt"] = np.asarray(encode_text(d["summary"], TARGET_TOKENS - 1) + [EOS_ID], dtype=np.int32)
        out.append(it)
    return out

_cache = OUTPUT_DIR / f"p6_inputs_{TOKENIZER_SHA256[:8]}_{SOURCE_STRATEGY}_{SOURCE_TOKENS}_{TARGET_TOKENS}_{sum(map(len, DATA.values()))}.pkl"
if _cache.exists():
    INPUTS = pickle.loads(_cache.read_bytes())
    print("Loaded cached model inputs.")
else:
    INPUTS = {s: build_inputs(DATA[s], SOURCE_STRATEGY, s != "test") for s in DATA}
    _cache.write_bytes(pickle.dumps(INPUTS, protocol=4))
_sl = np.array([len(x["src"]) for x in INPUTS["train"]]); _tl = np.array([len(x["tgt"]) for x in INPUTS["train"]])
print(f"source tokens: median {int(np.median(_sl))} (budget {SOURCE_TOKENS}) | summary tokens: median {int(np.median(_tl))} "
      f"| summaries cut at {TARGET_TOKENS} tokens: {np.mean(_tl >= TARGET_TOKENS):.0%}")
_ref_tok = np.array([len(encode_text(d["summary"])) for d in DATA["val"]])
print(f"full reference summaries (val): median {int(np.median(_ref_tok))} tokens")

class SummDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]

def collate(batch):
    B = len(batch)
    S = max(len(b["src"]) for b in batch)
    src = torch.full((B, S), PAD_ID, dtype=torch.long); smask = torch.zeros((B, S), dtype=torch.bool)
    for i, b in enumerate(batch):
        n = len(b["src"])
        sl = slice(S - n, S) if SOURCE_PAD_SIDE == "left" else slice(0, n)
        src[i, sl] = torch.from_numpy(b["src"].astype(np.int64)); smask[i, sl] = True
    out = dict(src=src, src_mask=smask, src_len=smask.sum(1))
    if "tgt" in batch[0]:
        P = len(PROMPT_IDS)
        Z = [PROMPT_IDS + b["tgt"].tolist() for b in batch]
        T = max(len(z) for z in Z) - 1
        inp = torch.full((B, T), PAD_ID, dtype=torch.long); lab = torch.full((B, T), -100, dtype=torch.long)
        tmask = torch.zeros((B, T), dtype=torch.bool)
        for i, z in enumerate(Z):
            z = torch.tensor(z)
            inp[i, :len(z) - 1] = z[:-1]; tmask[i, :len(z) - 1] = True
            lab[i, P - 1:len(z) - 1] = z[P:]
        out.update(tgt=inp, tgt_mask=tmask, labels=lab)
    return out

def to_device(batch):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

In [ ]:
def sinusoidal_relative_positions(max_distance, d_model, device, base=10_000.0):
    positions = torch.arange(max_distance + 1, device=device, dtype=torch.float32)
    inv_freq = torch.exp(-math.log(base) * torch.arange(0, d_model, 2, device=device, dtype=torch.float32) / d_model)
    angles = positions[:, None] * inv_freq[None, :]
    return torch.cat((angles.sin(), angles.cos()), dim=-1)

class RelativeMultiheadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout, rel_base=10_000.0):
        super().__init__()
        self.d_model, self.n_heads = d_model, n_heads
        self.head_dim = d_model // n_heads
        self.rel_base = rel_base
        self.attn_dropout = nn.Dropout(dropout)
        self.q_net = nn.Linear(d_model, d_model, bias=False)
        self.k_net = nn.Linear(d_model, d_model, bias=False)
        self.v_net = nn.Linear(d_model, d_model, bias=False)
        self.r_net = nn.Linear(d_model, d_model, bias=False)
        self.o_net = nn.Linear(d_model, d_model, bias=False)
        self.r_w_bias = nn.Parameter(torch.zeros(n_heads, self.head_dim))
        self.r_r_bias = nn.Parameter(torch.zeros(n_heads, self.head_dim))

    def _split_heads(self, x):
        b, t, _ = x.shape
        return x.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x, memory=None, key_mask=None):
        b, qlen, _ = x.shape
        mlen = 0 if memory is None else memory.size(1)
        kv_input = x if memory is None else torch.cat((memory.to(x.dtype), x), dim=1)
        klen = kv_input.size(1)
        q = self._split_heads(self.q_net(x))
        k = self._split_heads(self.k_net(kv_input))
        value = self._split_heads(self.v_net(kv_input))
        ac = torch.matmul((q + self.r_w_bias[None, :, None, :]).float(), k.float().transpose(-1, -2))
        rel_pos = sinusoidal_relative_positions(mlen + qlen - 1, self.d_model, x.device, self.rel_base)
        rel_keys = self._split_heads(self.r_net(rel_pos[None].to(self.r_net.weight.dtype)))[0].transpose(0, 1)
        bd_by_distance = torch.einsum("bhqd,rhd->bhqr", (q + self.r_r_bias[None, :, None, :]).float(), rel_keys.float())
        key_pos = torch.arange(-mlen, qlen, device=x.device)
        query_pos = torch.arange(qlen, device=x.device)
        rel_dist = (query_pos[:, None] - key_pos[None, :]).clamp_min(0)
        bd = torch.gather(bd_by_distance, dim=-1, index=rel_dist[None, None].expand(b, self.n_heads, qlen, klen))
        scores = (ac + bd) * (self.head_dim ** -0.5)
        allowed = (key_pos[None, :] <= query_pos[:, None])[None, None]
        if key_mask is not None:
            allowed = allowed & key_mask[:, None, None, :]
        scores = scores.masked_fill(~allowed, torch.finfo(scores.dtype).min)
        weights = self.attn_dropout(torch.softmax(scores, dim=-1).to(value.dtype))
        out = torch.matmul(weights, value).transpose(1, 2).contiguous().view(b, qlen, self.d_model)
        return self.o_net(out)

class FeedForward(nn.Module):
    def __init__(self, d_model, hidden, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, hidden), nn.GELU(), nn.Linear(hidden, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)

class TransformerXLBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_hidden, dropout, rel_base):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = RelativeMultiheadAttention(d_model, n_heads, dropout, rel_base)
        self.ff = FeedForward(d_model, ff_hidden, dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, memory=None, key_mask=None):
        x = x + self.resid_dropout(self.attn(self.norm1(x), memory, key_mask))
        return x + self.resid_dropout(self.ff(self.norm2(x)))

class TXLBackbone(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, ff_hidden, dropout, rel_base=10_000.0):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.embedding_dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([TransformerXLBlock(d_model, n_heads, ff_hidden, dropout, rel_base)
                                     for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)

    @property
    def blocks(self):
        return self.layers

    def forward(self, input_ids, mems=None, key_mask=None, return_hidden=False, grad_ckpt=False):
        h = self.embedding_dropout(self.token_embedding(input_ids))
        levels = [h]
        for i, layer in enumerate(self.layers):
            mem = None if mems is None else mems[i]
            if grad_ckpt:
                h = checkpoint(layer, h, mem, key_mask, use_reentrant=False)
            else:
                h = layer(h, mem, key_mask)
            levels.append(h)
        out = self.final_norm(h)
        return (out, levels) if return_hidden else out

def build_backbone(n_layers=None, layer_map=None, dropout=None):
    c = P1_CFG
    n = n_layers or c["layers"]
    bb = TXLBackbone(VOCAB_SIZE, c["d_model"], c["heads"], n, c["ff_hidden"],
                     BACKBONE_DROPOUT if dropout is None else dropout, c.get("rel_base", 10_000.0))
    sd = {k: v for k, v in P1_STATE.items() if not k.startswith("lm_head")}
    if layer_map is not None:
        assert len(layer_map) == n
        new_sd = {k: v for k, v in sd.items() if not k.startswith("layers.")}
        for new_i, old_i in enumerate(layer_map):
            pre = f"layers.{old_i}."
            for k, v in sd.items():
                if k.startswith(pre):
                    new_sd[f"layers.{new_i}." + k[len(pre):]] = v
        sd = new_sd
    missing, unexpected = bb.load_state_dict(sd, strict=False)
    if missing or unexpected:
        raise RuntimeError(f"Phase-1 checkpoint does not match backbone. missing={missing[:5]} unexpected={unexpected[:5]}")
    return bb

class LMHeadMixin:
    def _init_lm(self, emb):
        self._emb = [emb]
        w = P1_STATE.get("lm_head.weight")
        if w is not None and not (w.shape == emb.weight.shape and torch.equal(w, emb.weight.detach().cpu())):
            self.lm_weight = nn.Parameter(w.detach().clone().float(), requires_grad=False)
        else:
            self.lm_weight = None

    def project(self, h):
        W = self.lm_weight if self.lm_weight is not None else self._emb[0].weight
        return F.linear(h, W.to(h.dtype)).float()

    def head_modules(self):
        return []

    def forward(self, b):
        ctx = self.context(b["src"], b["src_mask"])
        return self.logits(ctx, b["tgt"], b["tgt_mask"])

def build_student_backbone(cfg, drop):
    n = cfg.get("student_layers") or P1_CFG["layers"]
    lm = [int(round(x)) for x in np.linspace(0, P1_CFG["layers"] - 1, n)]
    return build_backbone(n_layers=n, layer_map=lm, dropout=drop), lm

class SeqModel(LMHeadMixin, nn.Module):
    def __init__(self, backbone):
        nn.Module.__init__(self)
        self.backbone = backbone
        self._init_lm(backbone.token_embedding)
        self.grad_ckpt = False

    def context(self, src, src_mask):
        with torch.no_grad():
            _, lv = self.backbone(src, key_mask=src_mask, return_hidden=True)
        return dict(mems=[x.detach() for x in lv[:-1]], mask=src_mask)

    def logits(self, ctx, tgt, tgt_mask):
        h = self.backbone(tgt, mems=ctx["mems"], key_mask=torch.cat([ctx["mask"], tgt_mask], 1),
                          grad_ckpt=self.grad_ckpt and self.training)
        return self.project(h)

_probe = build_backbone()
P1_PARAMS = sum(p.numel() for p in _probe.parameters())
print(f"Phase-1 Transformer-XL: {P1_PARAMS/1e6:.2f}M parameters (tied LM head counted once)")
assert MIN_PARAMS <= P1_PARAMS <= MAX_PARAMS or SMOKE_TEST
del _probe

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout=0.0, rank_stabilised=False):
        super().__init__()
        self.base = base
        self.scale = alpha / (math.sqrt(r) if rank_stabilised else r)
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)

class LoRMAPlusLinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)
        y = y0 + self.scale * F.linear(F.linear(y0, self.A), self.B)
        return y if self.base.bias is None else y + self.base.bias

NF4_LEVELS = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
                           -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
                           0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
                           0.7229568362236023, 1.0])
QBLOCK = 64

def nf4_quantize(w):
    flat = w.detach().float().reshape(-1)
    pad = (-flat.numel()) % QBLOCK
    flat = torch.cat([flat, flat.new_zeros(pad)])
    blocks = flat.view(-1, QBLOCK)
    absmax = blocks.abs().max(1).values.clamp(min=1e-8)
    codes = (blocks / absmax[:, None]).unsqueeze(-1).sub(NF4_LEVELS).abs().argmin(-1).to(torch.uint8).view(-1)
    packed = (codes[0::2] | (codes[1::2] << 4)).contiguous()
    return packed, absmax.half(), pad

class QLoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.out_features, self.in_features = base.out_features, base.in_features
        packed, absmax, pad = nf4_quantize(base.weight)
        self.register_buffer("qweight", packed)
        self.register_buffer("absmax", absmax)
        self.register_buffer("levels", NF4_LEVELS.clone())
        self.pad = pad
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(), requires_grad=False)
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, self.in_features))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def dequantize(self, dtype):
        codes = torch.stack([self.qweight & 15, self.qweight >> 4], 1).view(-1).long()
        w = (self.levels[codes].view(-1, QBLOCK) * self.absmax.float()[:, None]).view(-1)
        if self.pad:
            w = w[:-self.pad]
        return w.view(self.out_features, self.in_features).to(dtype)

    def forward(self, x):

        y = F.linear(x, self.dequantize(torch.float32), self.bias)
        return y + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)

class AdaLoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r = base, r
        self.scale = alpha / r
        self.Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.E = nn.Parameter(torch.zeros(r))
        self.P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.Q) * self.E.to(x.dtype), self.P)

    def orth_penalty(self):
        I = torch.eye(self.r, device=self.Q.device)
        return (torch.norm(self.Q @ self.Q.T - I, p="fro") + torch.norm(self.P.T @ self.P - I, p="fro"))

class RankAllocator:
    def __init__(self, modules, cfg, total_steps):
        self.mods, self.cfg, self.total = modules, cfg, total_steps
        self.b1, self.b2 = cfg["beta1"], cfg["beta2"]
        self.init_budget = sum(m.r for m in modules)
        self.target_budget = cfg["target_rank"] * len(modules)
        self.ipt, self.unc = {}, {}
        self.final_keep = None
        self.budget = self.init_budget

    @torch.no_grad()
    def update_importance(self):
        for mi, m in enumerate(self.mods):
            for name in ("P", "E", "Q"):
                p = getattr(m, name)
                if p.grad is None:
                    continue
                cur = (p * p.grad).abs().float()
                key = (mi, name)
                if key not in self.ipt:
                    self.ipt[key], self.unc[key] = torch.zeros_like(cur), torch.zeros_like(cur)
                self.ipt[key].mul_(self.b1).add_((1 - self.b1) * cur)
                self.unc[key].mul_(self.b2).add_((1 - self.b2) * (cur - self.ipt[key]).abs())

    def _scores(self):
        out = []
        for mi, m in enumerate(self.mods):
            s = lambda n: self.ipt.get((mi, n), torch.zeros(1, device=m.E.device)) * \
                self.unc.get((mi, n), torch.zeros(1, device=m.E.device))
            sE = s("E").reshape(-1) if (mi, "E") in self.ipt else torch.zeros(m.r, device=m.E.device)
            sP = s("P").mean(0) if (mi, "P") in self.ipt else torch.zeros(m.r, device=m.E.device)
            sQ = s("Q").mean(1) if (mi, "Q") in self.ipt else torch.zeros(m.r, device=m.E.device)
            out.append(sE + sP + sQ)
        return out

    @torch.no_grad()
    def _apply(self, keep):
        for m, k in zip(self.mods, keep):
            m.E.masked_fill_(~k, 0.0)

    @torch.no_grad()
    def step(self, step):
        c, T = self.cfg, self.total
        t_init, t_final = int(c["t_init"] * T), int(c["t_final"] * T)
        if step <= t_init:
            return
        if step > T - t_final:
            if self.final_keep is None:
                self.final_keep = self._keep(self.target_budget)
            self._apply(self.final_keep)
            self.budget = self.target_budget
            return
        frac = 1 - (step - t_init) / max(1, T - t_final - t_init)
        self.budget = int((self.init_budget - self.target_budget) * frac ** 3 + self.target_budget)
        if step % c["delta_t"] == 0:
            self._apply(self._keep(self.budget))

    def _keep(self, budget):
        scores = self._scores()
        flat = torch.cat(scores)
        if budget >= flat.numel():
            return [torch.ones_like(s, dtype=torch.bool) for s in scores]
        keep = torch.zeros(flat.numel(), dtype=torch.bool, device=flat.device)
        keep[torch.topk(flat, budget).indices] = True
        return list(torch.split(keep, [s.numel() for s in scores]))

    def ranks(self):
        return [int((m.E != 0).sum()) for m in self.mods]

class HoulsbyAdapter(nn.Module):
    def __init__(self, base, bottleneck):
        super().__init__()
        self.base = base
        d = base.out_features
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        y = self.base(x)
        return y + self.up(F.gelu(self.down(y)))

ADAPTED_TYPES = (LoRALinear, LoRMAPlusLinear, QLoRALinear, AdaLoRALinear, HoulsbyAdapter)

def _resolve(block, path):
    parent = block
    parts = path.split(".")
    for p in parts[:-1]:
        parent = getattr(parent, p)
    return parent, parts[-1], getattr(parent, parts[-1])

def wrap_linears(backbone, targets, factory):
    n = 0
    for block in backbone.blocks:
        for path in targets:
            try:
                parent, name, leaf = _resolve(block, path)
            except AttributeError:
                continue
            if isinstance(leaf, nn.Linear):
                setattr(parent, name, factory(leaf)); n += 1
    return n

def is_norm(m):
    return isinstance(m, nn.LayerNorm) or "RMSNorm" in type(m).__name__

def block_param_ids(bb):
    return [{id(p) for p in b.parameters()} for b in bb.blocks]

def top_extras(bb):
    inside = set().union(*block_param_ids(bb))
    return [p for n, p in bb.named_parameters() if id(p) not in inside and "emb" not in n.split(".")[0]]

def configure_trainable(model, cfg):
    method = cfg["method"]
    bb = model.backbone
    for p in model.parameters():
        p.requires_grad = False
    save = set()
    if method == "zero_shot":
        pass
    elif method in ("full", "distill"):
        for p in model.parameters():
            p.requires_grad = True
    elif method in ("lora", "rslora"):
        for m in bb.modules():
            if isinstance(m, LoRALinear):
                m.A.requires_grad = m.B.requires_grad = True
    elif method == "lorma_plus":
        for m in bb.modules():
            if isinstance(m, LoRMAPlusLinear):
                m.A.requires_grad = m.B.requires_grad = True
    elif method == "adalora":
        for m in bb.modules():
            if isinstance(m, AdaLoRALinear):
                m.P.requires_grad = m.E.requires_grad = m.Q.requires_grad = True
    elif method == "qlora":
        for m in bb.modules():
            if isinstance(m, QLoRALinear):
                m.A.requires_grad = m.B.requires_grad = True
    elif method == "prefix":
        for p in bb.prefix.parameters():
            p.requires_grad = True
    elif method in ("houlsby", "pfeiffer"):
        for m in bb.modules():
            if isinstance(m, HoulsbyAdapter):
                for p in list(m.down.parameters()) + list(m.up.parameters()):
                    p.requires_grad = True
            elif is_norm(m):
                for p in m.parameters():
                    p.requires_grad = True
    elif method == "partial":
        for b in list(bb.blocks)[-cfg["top_layers"]:]:
            for p in b.parameters():
                p.requires_grad = True
        for p in top_extras(bb):
            p.requires_grad = True
    else:
        raise ValueError(method)
    for m in model.head_modules():
        for p in m.parameters():
            p.requires_grad = True
    save |= {n for n, p in model.named_parameters() if p.requires_grad}
    model.save_names = save

def param_groups(model, cfg):
    head_ids = {id(p) for m in model.head_modules() for p in m.parameters()}
    head = [p for p in model.parameters() if id(p) in head_ids]
    groups = [dict(params=head, lr=cfg.get("head_lr", cfg["lr"]), weight_decay=0.0)]
    decay = WEIGHT_DECAY if cfg["method"] in ("full", "distill", "partial") else 0.0
    bb = model.backbone
    if cfg["method"] == "partial":
        blocks = list(bb.blocks)[::-1]
        k = cfg["top_layers"]
        for depth, b in enumerate(blocks[:k]):
            groups.append(dict(params=list(b.parameters()), lr=cfg["lr"] * cfg["llrd"] ** depth, weight_decay=decay))
        groups.append(dict(params=top_extras(bb), lr=cfg["lr"], weight_decay=0.0))
    else:
        body = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
        if body:
            groups.append(dict(params=body, lr=cfg["lr"], weight_decay=decay))
    for g in groups:
        g["initial_lr"] = g["lr"]
    return [g for g in groups if g["params"]]

def make_controller(model, cfg, total_steps):
    if cfg["method"] == "adalora":
        mods = [m for m in model.backbone.modules() if isinstance(m, AdaLoRALinear)]
        return RankAllocator(mods, cfg, total_steps)
    return None

def extra_loss(model, cfg):
    if cfg["method"] != "adalora":
        return None
    mods = [m for m in model.backbone.modules() if isinstance(m, AdaLoRALinear)]
    return cfg["orth_reg"] * torch.stack([m.orth_penalty() for m in mods]).mean()

def build_seq_model(cfg):
    method = cfg["method"]
    drop = cfg["dropout"] if cfg.get("dropout") is not None else BACKBONE_DROPOUT
    if method == "distill":
        backbone, lm = build_student_backbone(cfg, drop)
        cfg["layer_map"] = lm
    else:
        backbone = build_backbone(dropout=drop)
    if method in ("lora", "rslora"):
        n = wrap_linears(backbone, LOWRANK_TARGETS, lambda l: LoRALinear(l, cfg["rank"], cfg["alpha"], 0.05, method == "rslora"))
        print(f"{method}: adapted {n} linear layers (rank {cfg['rank']})")
    elif method == "lorma_plus":
        n = wrap_linears(backbone, LOWRANK_TARGETS, lambda l: LoRMAPlusLinear(l, cfg["rank"], cfg["alpha"]))
        print(f"lorma_plus: adapted {n} linear layers")
    elif method == "adalora":
        n = wrap_linears(backbone, LOWRANK_TARGETS, lambda l: AdaLoRALinear(l, cfg["init_rank"], cfg["alpha"]))
        print(f"adalora: adapted {n} linear layers (rank {cfg['init_rank']} -> avg {cfg['target_rank']})")
    elif method == "houlsby":
        n = wrap_linears(backbone, ADAPTER_TARGETS, lambda l: HoulsbyAdapter(l, cfg["bottleneck"]))
        print(f"houlsby: {n} adapters (after attention outputs and FFNs, bottleneck {cfg['bottleneck']})")
    elif method == "pfeiffer":
        n = wrap_linears(backbone, FFN_OUT_TARGETS, lambda l: HoulsbyAdapter(l, cfg["bottleneck"]))
        print(f"pfeiffer: {n} adapters (after each FFN only, bottleneck {cfg['bottleneck']})")
    elif method == "qlora":
        n = wrap_linears(backbone, LOWRANK_TARGETS, lambda l: QLoRALinear(l, cfg["rank"], cfg["alpha"]))
        print(f"qlora: {n} linear layers stored in 4-bit NF4 + LoRA rank {cfg['rank']}")
    elif method == "prefix":
        if not hasattr(backbone, "add_prefix"):
            raise ValueError("prefix tuning is implemented for the multi-resolution memory model only")
        backbone.add_prefix(cfg["prefix_len"], cfg["prefix_hidden"])
        print(f"prefix: {cfg['prefix_len']} learned prefix states in each of {len(backbone.blocks)} layers")
    cfg["source_strategy"] = cfg.get("source_strategy") or SOURCE_STRATEGY
    model = SeqModel(backbone)
    model.grad_ckpt = USE_GRAD_CHECKPOINTING
    configure_trainable(model, cfg)
    if method in ("full", "distill") and model.lm_weight is not None:
        model.lm_weight.requires_grad = method == "distill" or method == "full"
    model.save_names = {n for n, p in model.named_parameters() if p.requires_grad}
    return model

def _adapter_checks():
    torch.manual_seed(0)
    base = nn.Linear(24, 32)
    x = torch.randn(3, 24)
    ref = base(x)
    for name, m in [("lora", LoRALinear(base, 4, 8)), ("lorma_plus", LoRMAPlusLinear(base, 4, 4)), ("adalora", AdaLoRALinear(base, 4, 8)),
                    ("houlsby", HoulsbyAdapter(base, 8))]:
        assert torch.allclose(m(x), ref, atol=1e-5), f"{name} not identity at init"
    q = QLoRALinear(base, 4, 8)
    rel = (q.dequantize(torch.float32) - base.weight).norm() / base.weight.norm()
    assert rel < 0.15, f"NF4 reconstruction error too high: {rel:.3f}"

    mods = [AdaLoRALinear(nn.Linear(16, 16), 6, 8) for _ in range(3)]
    for m in mods:
        m.E.data.uniform_(0.5, 1.0)
        for p in (m.P, m.E, m.Q):
            p.grad = torch.randn_like(p)
    ra = RankAllocator(mods, dict(beta1=0.85, beta2=0.85, target_rank=2, t_init=0.0, t_final=0.0, delta_t=1), 10)
    ra.update_importance()
    ra._apply(ra._keep(7))
    assert sum(ra.ranks()) == 7, ra.ranks()
    print(f"Adapter checks passed (LoRMA+, AdaLoRA, Houlsby = identity at init; AdaLoRA budget masking works; "
          f"NF4 4-bit relative weight error {rel:.3f}).")

_adapter_checks()

In [ ]:
_BANNED_ALWAYS = [PAD_ID, UNK_ID, BOS_ID]

@torch.no_grad()
def generate(model, items, max_new=None, min_new=None, batch=None, desc="generate"):
    model.eval()
    max_new = max_new or TARGET_TOKENS
    min_new = min(GEN_MIN_TOKENS if min_new is None else min_new, max_new - 1)
    batch = batch or GEN_BATCH
    order = sorted(range(len(items)), key=lambda i: len(items[i]["src"]))
    out = [None] * len(items)
    n = NO_REPEAT_NGRAM
    for k in tqdm(range(0, len(order), batch), desc=desc, leave=False):
        idx = order[k:k + batch]
        b = to_device(collate([{"src": items[i]["src"]} for i in idx]))
        B = len(idx)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
            ctx = model.context(b["src"], b["src_mask"])
        tgt = torch.tensor([PROMPT_IDS] * B, dtype=torch.long, device=DEVICE)
        seen = torch.zeros(B, VOCAB_SIZE, dtype=torch.bool, device=DEVICE)
        done = torch.zeros(B, dtype=torch.bool, device=DEVICE)
        gen = [[] for _ in range(B)]
        grams = [defaultdict(set) for _ in range(B)]
        for t in range(max_new):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
                lg = model.logits(ctx, tgt, torch.ones_like(tgt, dtype=torch.bool))[:, -1].float()
            if REPETITION_PENALTY != 1.0:
                pen = torch.where(lg > 0, lg / REPETITION_PENALTY, lg * REPETITION_PENALTY)
                lg = torch.where(seen, pen, lg)
            lg[:, _BANNED_ALWAYS] = -1e9
            if t < min_new:
                lg[:, EOS_ID] = -1e9
            if n > 0 and t >= n - 1:
                for r in range(B):
                    if not done[r]:
                        ban = grams[r].get(tuple(gen[r][-(n - 1):]))
                        if ban:
                            lg[r, list(ban)] = -1e9
            nxt = lg.argmax(-1)
            nxt = torch.where(done, torch.full_like(nxt, PAD_ID), nxt)
            tgt = torch.cat([tgt, nxt[:, None]], 1)
            for r, tok in enumerate(nxt.tolist()):
                if done[r]:
                    continue
                gen[r].append(tok)
                if n > 0 and len(gen[r]) >= n:
                    grams[r][tuple(gen[r][-n:-1])].add(tok)
            seen[torch.arange(B, device=DEVICE), nxt] = True
            done |= nxt == EOS_ID
            if bool(done.all()):
                break
        for r, i in enumerate(idx):
            ids = [x for x in gen[r] if x != EOS_ID]
            out[i] = postprocess(decode_ids(ids))
    return out

def postprocess(text):
    sents, seen = [], set()
    for s in split_sentences(text):
        key = re.sub(r"\W+", " ", s.lower()).strip()
        if key and key not in seen:
            seen.add(key); sents.append(s)
    if len(sents) > 1 and not re.search(r"[.?!]\s*$", sents[-1]):
        sents = sents[:-1]
    return " ".join(sents).strip() or text.strip()

def _words(t):
    return re.findall(r"[a-z0-9]+", t.lower())

def _ngrams(w, n):
    return Counter(tuple(w[i:i + n]) for i in range(len(w) - n + 1))

def _f(p, r):
    return 2 * p * r / (p + r) if p + r else 0.0

def rouge_n(hyp, ref, n):
    h, r = _ngrams(hyp, n), _ngrams(ref, n)
    ov = sum((h & r).values())
    return _f(ov / max(1, sum(h.values())), ov / max(1, sum(r.values())))

def rouge_l(hyp, ref):
    if not hyp or not ref:
        return 0.0
    ref_pos = defaultdict(list)
    for j, w in enumerate(ref):
        ref_pos[w].append(j)
    prev = np.zeros(len(ref) + 1, dtype=np.int32)
    for w in hyp:
        cur = prev.copy()
        js = ref_pos.get(w)
        if js:
            for j in js:
                cur[j + 1] = prev[j] + 1
        np.maximum.accumulate(cur, out=cur)
        prev = cur
    lcs = int(prev[-1])
    return _f(lcs / len(hyp), lcs / len(ref))

_ENTITY_RE = re.compile(r"\b(?:section|sec|article|art|rule|order)\.?\s*\d+[a-z]?\b|\b\d[\d./-]*\b", re.I)

def entities(t):
    return {re.sub(r"\s+", " ", m.group(0).lower()).replace("sec.", "section").replace("art.", "article")
            for m in _ENTITY_RE.finditer(t)}

def summary_metrics(hyps, refs, sources):
    rows = []
    for h, r, s in zip(hyps, refs, sources):
        hw, rw, sw = _words(h), _words(r), _words(s)
        tri = _ngrams(hw, 3)
        bi_src = set(_ngrams(sw, 2))
        he, re_, se = entities(h), entities(r), entities(s)
        rows.append(dict(rouge1=rouge_n(hw, rw, 1), rouge2=rouge_n(hw, rw, 2), rougeL=rouge_l(hw, rw),
                         words=len(hw), ref_words=len(rw),
                         repeated_trigrams=1 - len(tri) / max(1, sum(tri.values())),
                         novel_bigrams=float(np.mean([g not in bi_src for g in _ngrams(hw, 2)])) if len(hw) > 1 else 0.0,
                         entity_coverage=len(he & re_) / len(re_) if re_ else np.nan,
                         entity_support=len(he & se) / len(he) if he else np.nan))
    keys = rows[0].keys() if rows else []
    return {k: float(np.nanmean([r[k] for r in rows])) for k in keys}

In [ ]:
TRAINED_MODELS, RESULTS = {}, {}

def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if (p.requires_grad or not trainable_only))

def logical_params(module):
    q = sum(m.out_features * m.in_features for m in module.modules() if isinstance(m, QLoRALinear))
    return count_params(module) + q

def make_cfg(name):
    cfg = dict(EXPERIMENT_DEFAULTS)
    cfg.update(copy.deepcopy(EXPERIMENTS[name]))
    cfg["name"] = name
    cfg["epochs"] = EPOCHS if cfg.get("epochs") is None else cfg["epochs"]
    return cfg

def saved_state(model):
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in model.save_names}

def load_trained(name):
    run_dir = OUTPUT_DIR / "runs" / name
    cfg = json.loads((run_dir / "cfg.json").read_text())
    model = build_seq_model(cfg)
    state = torch.load(run_dir / "best_state.pt", map_location="cpu")
    model.load_state_dict(state, strict=False)
    return model

def write_submission(run_dir, summaries):
    sub = [{"id": d["id"], "summary": s} for d, s in zip(DATA["test"], summaries)]
    assert [s["id"] for s in sub] == test_ids and all(isinstance(s["summary"], str) for s in sub)
    pj = Path(run_dir) / "prediction.json"
    pj.write_text(json.dumps(sub, ensure_ascii=False), encoding="utf-8")
    zp = Path(run_dir) / "submission.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    return zp

def token_loss(logits, labels):
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100)

def kd_loss(s_logits, t_logits, labels, cfg):
    m = labels != -100
    T = cfg["kd_temperature"]
    s, t = s_logits[m].float() / T, t_logits[m].float() / T
    kd = F.kl_div(F.log_softmax(s, -1), F.log_softmax(t, -1), log_target=True, reduction="batchmean") * T * T
    return cfg["kd_alpha"] * token_loss(s_logits, labels) + (1 - cfg["kd_alpha"]) * kd

@torch.no_grad()
def val_loss(model):
    model.eval()
    tot, n = 0.0, 0
    for b in DataLoader(SummDataset(INPUTS["val"]), batch_size=BATCH_DOCS, collate_fn=collate):
        b = to_device(b)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
            lg = model(b)
        k = int((b["labels"] != -100).sum())
        tot += float(token_loss(lg, b["labels"])) * k; n += k
    return tot / max(1, n)

VAL_GEN = INPUTS["val"][:VAL_GEN_DOCS]
VAL_REFS = [d["summary"] for d in DATA["val"][:VAL_GEN_DOCS]]
VAL_SRCS = [d["document"] for d in DATA["val"][:VAL_GEN_DOCS]]

def run_experiment(name, teacher=None):
    cfg = make_cfg(name)
    run_dir = OUTPUT_DIR / "runs" / name
    run_dir.mkdir(parents=True, exist_ok=True)
    if RESUME_COMPLETED_RUNS and (run_dir / "results.json").exists():
        RESULTS[name] = json.loads((run_dir / "results.json").read_text())
        print(f"[{name}] already finished (val ROUGE-L {RESULTS[name]['val']['rougeL']:.4f}) - skipping")
        return RESULTS[name]
    print("=" * 100); print(f"{name}: {cfg['description']}"); print("=" * 100)
    seed_everything(SEED)
    is_kd = cfg["method"] == "distill"
    model = build_seq_model(cfg).to(DEVICE)
    n_bb, n_total = logical_params(model.backbone), logical_params(model)
    n_train = sum(p.numel() for n, p in model.named_parameters() if n in model.save_names)
    print(f"backbone (+adapters) {n_bb/1e6:.2f}M | with LM head {n_total/1e6:.2f}M | trainable {n_train/1e6:.3f}M "
          f"({100*n_train/max(n_total,1):.2f}%) | source {SOURCE_TOKENS} ({cfg['source_strategy']}) -> summary {TARGET_TOKENS}")
    assert MIN_PARAMS <= n_bb <= MAX_PARAMS or SMOKE_TEST, \
        f"backbone {n_bb/1e6:.1f}M outside the {MIN_PARAMS/1e6:.0f}M-{MAX_PARAMS/1e6:.0f}M limit"
    if is_kd:
        assert teacher is not None, "distillation needs a teacher"
        teacher.to(DEVICE).eval()
        for p in teacher.parameters():
            p.requires_grad = False
        print(f"teacher: {logical_params(teacher.backbone)/1e6:.1f}M -> student {n_bb/1e6:.1f}M (layers {cfg['layer_map']})")
    train_items = INPUTS["train"]
    if cfg["source_strategy"] != SOURCE_STRATEGY:
        train_items = build_inputs(DATA["train"], cfg["source_strategy"], True)
    history, t0, best, best_ep, bad = [], time.time(), float("inf"), 0, 0
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    if cfg["epochs"] > 0 and n_train > 0:
        groups = param_groups(model, cfg)
        optim = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-8)
        steps_per_epoch = max(1, math.ceil(len(train_items) / (BATCH_DOCS * GRAD_ACCUM)))
        total = steps_per_epoch * cfg["epochs"]
        warm = max(1, int(WARMUP_FRACTION * total))
        sched = torch.optim.lr_scheduler.LambdaLR(optim, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
        controller = make_controller(model, cfg, total)
        use_amp = cfg["use_amp"] and DEVICE.type == "cuda"
        scaler = make_grad_scaler(use_amp)
        params = [p for g in groups for p in g["params"]]
        gstep, out_of_time = 0, False
        print(f"{len(train_items):,} training documents x {cfg['epochs']} epochs | optimizer steps {total} | "
              f"time budget {MAX_TRAIN_MINUTES} min")

        def step():
            nonlocal gstep
            if scaler is not None:
                scaler.unscale_(optim)
            if controller is not None:
                controller.update_importance()
            torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
            if scaler is None:
                optim.step()
            else:
                scaler.step(optim); scaler.update()
            sched.step(); optim.zero_grad(set_to_none=True)
            gstep += 1
            if controller is not None:
                controller.step(gstep)

        for ep in range(1, cfg["epochs"] + 1):
            loader = DataLoader(SummDataset(train_items), batch_size=BATCH_DOCS, shuffle=True, collate_fn=collate,
                                num_workers=2 if DEVICE.type == "cuda" else 0)
            model.train()
            run_loss, n_loss, micro, te = 0.0, 0, 0, time.time()
            optim.zero_grad(set_to_none=True)
            for b in tqdm(loader, desc=f"{name} ep{ep}", leave=False):
                b = to_device(b)
                try:
                    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                        lg = model(b)
                        if is_kd:
                            with torch.no_grad():
                                tl = teacher(b)
                            loss = kd_loss(lg, tl, b["labels"], cfg)
                        else:
                            loss = token_loss(lg.float(), b["labels"])
                        reg = extra_loss(model, cfg)
                        if reg is not None:
                            loss = loss + reg
                    if not torch.isfinite(loss):
                        optim.zero_grad(set_to_none=True); continue
                    (loss / GRAD_ACCUM).backward() if scaler is None else scaler.scale(loss / GRAD_ACCUM).backward()
                except torch.cuda.OutOfMemoryError:
                    optim.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                    if not model.grad_ckpt:
                        model.grad_ckpt = True
                        print("    CUDA out of memory -> gradient checkpointing switched on")
                    continue
                micro += 1; run_loss += float(loss); n_loss += 1
                if micro % GRAD_ACCUM == 0:
                    step()
                if micro == 20 * GRAD_ACCUM and ep == 1:
                    print(f"    ~{(time.time() - te) / micro * len(loader) / 60:.0f} min per epoch at the current speed")
                if time.time() - t0 > MAX_TRAIN_MINUTES * 60:
                    out_of_time = True
                    print(f"    time budget of {MAX_TRAIN_MINUTES} min reached in epoch {ep} ({micro}/{len(loader)} batches)")
                    break
            if micro % GRAD_ACCUM:
                step()
            vl = val_loss(model)
            history.append(dict(epoch=ep, train_loss=run_loss / max(1, n_loss), val_loss=vl, val_ppl=math.exp(min(vl, 20)),
                                elapsed_s=time.time() - t0))
            print(f"  epoch {ep} | train loss {history[-1]['train_loss']:.4f} | val loss {vl:.4f} (ppl {math.exp(min(vl, 20)):.1f}) "
                  f"| {(time.time()-t0)/60:.1f} min")
            if vl < best - 1e-4:
                best, best_ep, bad = vl, ep, 0
                torch.save(saved_state(model), run_dir / "best_state.pt")
            else:
                bad += 1
                if bad >= cfg["patience"]:
                    print(f"  early stop (no improvement for {cfg['patience']} epochs)")
                    break
            if out_of_time:
                break
        model.load_state_dict(torch.load(run_dir / "best_state.pt", map_location=DEVICE), strict=False)
    else:
        torch.save(saved_state(model), run_dir / "best_state.pt")
        best = val_loss(model)
    train_time = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 1024**3 if DEVICE.type == "cuda" else 0.0

    val_items = VAL_GEN if cfg["source_strategy"] == SOURCE_STRATEGY else \
        build_inputs(DATA["val"][:VAL_GEN_DOCS], cfg["source_strategy"], True)
    tg = time.time()
    hyps = generate(model, val_items, desc=f"{name} val")
    gen_s = (time.time() - tg) / max(1, len(val_items))
    val = summary_metrics(hyps, VAL_REFS, VAL_SRCS)
    print(f"[{name}] best epoch {best_ep} | val loss {best:.4f} | ROUGE-1/2/L {val['rouge1']:.4f}/{val['rouge2']:.4f}/"
          f"{val['rougeL']:.4f} | {val['words']:.0f} words (ref {val['ref_words']:.0f}) | repeated 3-grams "
          f"{val['repeated_trigrams']:.1%} | entity coverage {val['entity_coverage']:.1%} | {gen_s:.2f} s/summary")
    print(f"   example: {hyps[0][:400]}")
    save_json(run_dir / "val_summaries.json", [{"id": d["id"], "summary": h} for d, h in zip(DATA["val"], hyps)])
    out = dict(name=name, method=cfg["method"], description=cfg["description"], cfg=cfg,
               params_backbone=n_bb, params_total=n_total, params_trainable=n_train,
               trainable_pct=100 * n_train / max(1, n_total), train_time_s=train_time, best_epoch=best_ep,
               epochs_run=len(history), peak_mem_gb=peak, val_loss=best, val=val, gen_s_per_summary=gen_s,
               state_mb=(run_dir / "best_state.pt").stat().st_size / 1024**2, history=history)
    if WRITE_SUBMISSIONS:
        test_items = INPUTS["test"] if cfg["source_strategy"] == SOURCE_STRATEGY else \
            build_inputs(DATA["test"], cfg["source_strategy"], False)
        summ = generate(model, test_items, desc=f"{name} test")
        out["submission_zip"] = str(write_submission(run_dir, summ))
        out["test_avg_words"] = float(np.mean([len(s.split()) for s in summ]))
        print(f"        submission: {out['submission_zip']} ({out['test_avg_words']:.0f} words per summary)")
    save_json(run_dir / "cfg.json", cfg)
    save_json(run_dir / "results.json", out)
    RESULTS[name] = out
    model.cpu(); TRAINED_MODELS[name] = model
    if teacher is not None:
        teacher.cpu()
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return out

In [ ]:
for _p in sorted((OUTPUT_DIR / "runs").glob("*/results.json")):
    if _p.parent.name in EXPERIMENTS:
        RESULTS[_p.parent.name] = json.loads(_p.read_text())

TEACHER_CANDIDATES = [n for n, c in EXPERIMENTS.items() if c["method"] not in ("distill", "zero_shot")]

def pick_teacher():
    done = {n: RESULTS[n]["val"]["rougeL"] for n in TEACHER_CANDIDATES if n in RESULTS}
    if not done:
        raise RuntimeError("Distillation needs a finished teacher: run the other experiments first.")
    name = max(done, key=done.get)
    return name, TRAINED_MODELS.get(name) or load_trained(name)

for name in RUN_LIST:
    teacher = None
    if EXPERIMENTS[name]["method"] == "distill" and not (OUTPUT_DIR / "runs" / name / "results.json").exists():
        tname, teacher = pick_teacher()
        EXPERIMENTS[name]["teacher_name"] = tname
        print(f"Teacher for {name}: {tname} (val ROUGE-L {RESULTS[tname]['val']['rougeL']:.4f})")
    run_experiment(name, teacher)
    gc.collect()
print("\nFinished:", [n for n in RUN_LIST if n in RESULTS])

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 200)
ORDER = [n for n in EXPERIMENTS if n in RESULTS]
rows = []
for n in ORDER:
    r = RESULTS[n]; v = r["val"]
    rows.append(dict(experiment=n, method=r["method"], source=r["cfg"]["source_strategy"],
                     backbone_M=r["params_backbone"] / 1e6, trainable_M=r["params_trainable"] / 1e6,
                     trainable_pct=r["trainable_pct"], val_loss=r["val_loss"], ROUGE_1=v["rouge1"], ROUGE_2=v["rouge2"],
                     ROUGE_L=v["rougeL"], words=v["words"], ref_words=v["ref_words"], repeated_3grams=v["repeated_trigrams"],
                     novel_bigrams=v["novel_bigrams"], entity_coverage=v["entity_coverage"], entity_support=v["entity_support"],
                     train_min=r["train_time_s"] / 60, peak_mem_GB=r["peak_mem_gb"], s_per_summary=r["gen_s_per_summary"],
                     ckpt_MB=r["state_mb"]))
summary = pd.DataFrame(rows).set_index("experiment")
summary.to_csv(OUTPUT_DIR / f"phase6_{ARCH_NAME}_results.csv")
print(f"Validation ({len(VAL_GEN)} held-out training documents). BERTScore is computed by Codabench; ROUGE here is a "
      "from-scratch re-implementation used for model selection.")
print("entity_coverage = share of the reference's numbers / section references that the summary contains;")
print("entity_support  = share of the summary's numbers / section references that also occur in the judgment (hallucination proxy).")
display(summary.round(4))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
summary[["ROUGE_1", "ROUGE_2", "ROUGE_L"]].plot.bar(ax=ax[0]); ax[0].set_title("Validation ROUGE F1")
ax[0].tick_params(axis="x", rotation=30)
for n in ORDER:
    h = RESULTS[n]["history"]
    if h:
        ax[1].plot([x["epoch"] for x in h], [x["val_loss"] for x in h], marker="o", label=n)
ax[1].set_title("Validation loss per epoch"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "phase6_overview.png", dpi=130); plt.show()

best_name = max([n for n in ORDER if RESULTS[n]["method"] != "zero_shot"] or ORDER, key=lambda n: RESULTS[n]["val"]["rougeL"])
print(f"Best run: {best_name} (val ROUGE-L {RESULTS[best_name]['val']['rougeL']:.4f})")
vs = json.loads((OUTPUT_DIR / "runs" / best_name / "val_summaries.json").read_text())
for i in range(min(2, len(vs))):
    print(f"\n--- {vs[i]['id']} ---\nGENERATED: {vs[i]['summary'][:700]}\nREFERENCE: {DATA['val'][i]['summary'][:700]}")

STUDY = {}
if RUN_LENGTH_STUDY or RUN_SOURCE_STUDY:
    model = (TRAINED_MODELS.get(best_name) or load_trained(best_name)).to(DEVICE)
    if RUN_LENGTH_STUDY:
        for L in sorted({max(GEN_MIN_TOKENS + 8, TARGET_TOKENS // 3), TARGET_TOKENS // 2 + 32, TARGET_TOKENS}):
            h = generate(model, VAL_GEN, max_new=L, min_new=min(GEN_MIN_TOKENS, L - 8), desc=f"length {L}")
            STUDY[f"length cap {L} tokens"] = summary_metrics(h, VAL_REFS, VAL_SRCS)
    if RUN_SOURCE_STUDY:
        for strat in [s for s in ("salient", "head", "head+tail") if s != RESULTS[best_name]["cfg"]["source_strategy"]]:
            items = build_inputs(DATA["val"][:VAL_GEN_DOCS], strat, True)
            h = generate(model, items, desc=f"source {strat}")
            STUDY[f"source = {strat} (inference only)"] = summary_metrics(h, VAL_REFS, VAL_SRCS)
        STUDY[f"source = {RESULTS[best_name]['cfg']['source_strategy']} (trained)"] = RESULTS[best_name]["val"]
    model.cpu()
    st = pd.DataFrame(STUDY).T[["rouge1", "rouge2", "rougeL", "words", "repeated_trigrams", "entity_coverage", "entity_support"]]
    print(f"\nGeneration length and long-document input with {best_name}:")
    display(st.round(4))
    st.to_csv(OUTPUT_DIR / "phase6_length_source_study.csv")

print("\n--- Key comparisons for the written analysis ---")
ref_name = best_name
ref = RESULTS[ref_name]["val"]["rougeL"]
for n in ORDER:
    r = RESULTS[n]
    if n != ref_name:
        print(f"{n:28s}: ROUGE-L {r['val']['rougeL']:.4f} = {r['val']['rougeL']/max(ref, 1e-9):.1%} of {ref_name} "
              f"with {r['trainable_pct']:.2f}% trainable")
if "E4_distill" in RESULTS:
    r = RESULTS["E4_distill"]; tn = r["cfg"].get("teacher_name")
    if tn in RESULTS:
        t = RESULTS[tn]
        print(f"KD: student {r['params_backbone']/1e6:.1f}M ROUGE-L {r['val']['rougeL']:.4f} vs teacher {tn} "
              f"{t['params_backbone']/1e6:.1f}M {t['val']['rougeL']:.4f} | {r['gen_s_per_summary']:.2f} vs "
              f"{t['gen_s_per_summary']:.2f} s/summary | peak memory {r['peak_mem_gb']:.1f} vs {t['peak_mem_gb']:.1f} GB")

In [ ]:
SUB_DIR = OUTPUT_DIR / "submissions"
SUB_DIR.mkdir(exist_ok=True)
print(f"{'experiment':30s} {'val ROUGE-L':>11s}  zip")
for n in ORDER:
    zp = OUTPUT_DIR / "runs" / n / "submission.zip"
    if not zp.exists():
        continue
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"], z.namelist()
        sub = json.loads(z.read("prediction.json"))
    assert isinstance(sub, list) and [s["id"] for s in sub] == test_ids
    assert all(set(s) == {"id", "summary"} and isinstance(s["summary"], str) for s in sub)
    empty = sum(not s["summary"].strip() for s in sub)
    dest = SUB_DIR / f"phase6_{ARCH_NAME}_{n}.zip"
    shutil.copy(zp, dest)
    print(f"{n:30s} {RESULTS[n]['val']['rougeL']:11.4f}  {dest}" + (f"  ({empty} empty summaries!)" if empty else ""))
print("\nEach zip: prediction.json at the root = a JSON list of {\"id\", \"summary\"}, one per test document")
print("(Option A of the Phase-6 format). Upload on Codabench; your best four must come from different Phase-1 models.")